# Loan Performance Intelligence Engine — End-to-End Walkthrough

This notebook demonstrates the key outputs from our pipeline. Before running this notebook, ensure you have executed `python run_all.py` at the project root so all reports and data files are generated.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
import json
import os

ROOT_DIR = "../"
REPORTS_DIR = os.path.join(ROOT_DIR, "reports")
DATA_DIR = os.path.join(ROOT_DIR, "data/processed")

## 1. Data Intelligence & Profiling
Let's look at the top validation rule violations detected in the raw data.

In [ ]:
try:
    violations = pd.read_csv(os.path.join(DATA_DIR, "validation_violations.csv"))
    display(violations.head())
except FileNotFoundError:
    print("Validation violations file not found. Run python run_all.py first.")

## 2. Time-Aware Split and Leakage Proof
Our splits ensure no data leakage. Let's verify the months in the test set.

In [ ]:
try:
    test_data = pd.read_csv(os.path.join(ROOT_DIR, "data/raw/loan_monthly_performance_test.csv"))
    print("Test set months:", test_data["month_index"].unique())
except FileNotFoundError:
    print("Test set not found.")

## 3. Model Metrics
A summary of our binary prediction models.

In [ ]:
try:
    with open(os.path.join(DATA_DIR, "binary_model_metrics.json")) as f:
        metrics = json.load(f)
    
    df_metrics = pd.DataFrame([
        {"Target": tgt, "Model": model, **res}
        for tgt, models in metrics.items()
        for model, res in models.items() if "test" in model
    ])
    display(df_metrics)
except Exception as e:
    print("Metrics not found:", e)

## 4. Anomaly Detection
Top flagged exceptions using our hybrid rules+ML approach.

In [ ]:
try:
    anomalies = pd.read_csv(os.path.join(DATA_DIR, "anomaly_scores.csv"))
    flagged = anomalies[anomalies["exception_flag"] == 1].sort_values("anomaly_score", ascending=False)
    display(flagged[["loan_id", "anomaly_score", "predicted_exception_type", "top_drivers"]].head(10))
except Exception as e:
    print("Anomalies not found:", e)

## 5. Final Submission
The output template to submit.

In [ ]:
try:
    sub = pd.read_csv(os.path.join(ROOT_DIR, "submission/submission.csv"))
    display(sub.head())
except Exception as e:
    print("Submission not found:", e)